# Crypto Price Data Collection & Alignment

For each crypto-related tweet, capture price data at:
- T-24h: price_before_24h (baseline)
- T-1h: price_before_1h (just before tweet)
- T: price_at_tweet (moment of tweet)
- T+1h: price_after_1h (immediate reaction)
- T+24h: price_after_24h (short-term impact)
- T+7d: price_after_7d (medium-term impact)

In [15]:
import pandas as pd
import numpy as np
import requests
import time
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
from tqdm import tqdm
from pathlib import Path

load_dotenv()

CRYPTOCOMPARE_API_KEY = os.getenv("CRYPTOCOMPARE_API_KEY")
BASE_URL = "https://min-api.cryptocompare.com/data/v2"

# Token symbols (CryptoCompare uses standard symbols)
SUPPORTED_TOKENS = [
    "BTC", "ETH", "DOGE", "SHIB", "FLOKI",
    "SOL", "XRP", "ADA", "BNB", "LINK",
    "MATIC", "FTT", "BABYDOGE"
]

# These are categories, not tokens - filter them out
INVALID_TOKENS = ["DEFI", "NFT", "CRYPTO", "WEB3", "BLOCKCHAIN"]

# Default token for Musk's vague crypto tweets (emoji-only, "to the moon", etc.)
DEFAULT_TOKEN = "DOGE"

# Rate limiting
CALLS_PER_MINUTE = 50
CALL_INTERVAL = 60 / CALLS_PER_MINUTE


def get_headers():
    """Get API headers."""
    headers = {"accept": "application/json"}
    if CRYPTOCOMPARE_API_KEY:
        headers["authorization"] = f"Apikey {CRYPTOCOMPARE_API_KEY}"
    return headers


def test_api_connection():
    """Test API connection."""
    print("\nTesting CryptoCompare API connection...")
    print(f"  API Key set: {'Yes' if CRYPTOCOMPARE_API_KEY else 'No'}")

    url = "https://min-api.cryptocompare.com/data/v2/histoday"
    params = {"fsym": "BTC", "tsym": "USD", "limit": 1}

    try:
        response = requests.get(url, headers=get_headers(), params=params)
        data = response.json()

        if data.get("Response") == "Success":
            print("  ✓ API connection OK")
            return True
        else:
            print(f"  ✗ Error: {data.get('Message', 'Unknown error')}")
            return False
    except Exception as e:
        print(f"  ✗ Connection error: {e}")
        return False


def fetch_daily_prices(symbol: str, to_ts: int, limit: int = 2000) -> pd.DataFrame:
    """
    Fetch historical daily prices.

    Args:
        symbol: Token symbol (e.g., 'DOGE')
        to_ts: End timestamp (unix)
        limit: Number of days to fetch (max 2000)

    Returns:
        DataFrame with columns: [timestamp, price, volume]
    """
    url = f"{BASE_URL}/histoday"
    params = {
        "fsym": symbol,
        "tsym": "USD",
        "limit": limit,
        "toTs": to_ts,
    }

    try:
        response = requests.get(url, headers=get_headers(), params=params)
        data = response.json()

        if data.get("Response") != "Success":
            print(f"  API Error: {data.get('Message', 'Unknown')}")
            return pd.DataFrame()

        records = data.get("Data", {}).get("Data", [])
        if not records:
            return pd.DataFrame()

        df = pd.DataFrame(records)
        df['timestamp'] = pd.to_datetime(df['time'], unit='s', utc=True)
        df['price'] = df['close']
        df['volume'] = df['volumeto']

        return df[['timestamp', 'price', 'volume']]

    except Exception as e:
        print(f"  Error fetching {symbol}: {e}")
        return pd.DataFrame()


def fetch_full_history(symbol: str, min_date: datetime, max_date: datetime) -> pd.DataFrame:
    """
    Fetch full price history for a token.

    Args:
        symbol: Token symbol
        min_date: Start date
        max_date: End date

    Returns:
        DataFrame with timestamp, price, volume
    """
    all_data = []

    # Fetch in chunks of 2000 days if needed
    current_end = max_date + timedelta(days=8)  # Buffer for T+7d

    while current_end > min_date - timedelta(days=2):
        to_ts = int(current_end.timestamp())

        df = fetch_daily_prices(symbol, to_ts, limit=2000)

        if df.empty:
            break

        all_data.append(df)

        # Move window back
        earliest = df['timestamp'].min()
        if earliest <= min_date - timedelta(days=2):
            break
        current_end = earliest - timedelta(days=1)

        time.sleep(CALL_INTERVAL)

    if not all_data:
        return pd.DataFrame()

    # Combine and deduplicate
    combined = pd.concat(all_data, ignore_index=True)
    combined = combined.drop_duplicates(subset='timestamp').sort_values('timestamp')

    return combined.reset_index(drop=True)


def fetch_all_token_prices(tokens: list, min_date: datetime, max_date: datetime) -> dict:
    """
    Fetch historical prices for all tokens.
    """
    token_prices = {}

    for token in tqdm(tokens, desc="Fetching token prices"):
        token_upper = token.upper()

        print(f"\nFetching {token_upper}...")

        df = fetch_full_history(token_upper, min_date, max_date)

        if not df.empty:
            # Filter out zero prices (token didn't exist yet)
            df = df[df['price'] > 0]
            if not df.empty:
                print(f"  Got {len(df):,} data points from {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
                token_prices[token_upper] = df
            else:
                print(f"  No valid price data for {token_upper}")
        else:
            print(f"  No data for {token_upper}")

        time.sleep(CALL_INTERVAL)

    return token_prices


def get_price_at_time(price_df: pd.DataFrame, target_time: pd.Timestamp, max_gap_hours: int = 24) -> float:
    """
    Get price closest to target time.
    """
    if price_df.empty:
        return None

    if target_time.tzinfo is None:
        target_time = target_time.tz_localize('UTC')

    time_diffs = abs(price_df['timestamp'] - target_time)
    min_idx = time_diffs.idxmin()
    min_diff = time_diffs[min_idx]

    if min_diff > timedelta(hours=max_gap_hours):
        return None

    return price_df.loc[min_idx, 'price']


def extract_price_features(row: pd.Series, token_prices: dict) -> dict:
    """
    Extract price features for a single tweet.
    """
    features = {
        'price_token': None,
        'price_before_24h': None,
        'price_before_1h': None,
        'price_at_tweet': None,
        'price_after_1h': None,
        'price_after_24h': None,
        'price_after_7d': None,
        'volume_at_tweet': None,
        'pct_change_1h': None,
        'pct_change_24h': None,
        'pct_change_7d': None,
    }

    tweet_time = pd.to_datetime(row['createdAt'], utc=True)

    tokens = row.get('crypto_tokens', [])

    if tokens is None or (isinstance(tokens, float) and pd.isna(tokens)):
        tokens = []
    elif isinstance(tokens, str):
        try:
            tokens = eval(tokens)
        except:
            tokens = []

    if not isinstance(tokens, list):
        tokens = []

    # Filter out invalid tokens (categories like DEFI, NFT)
    tokens = [t.upper() for t in tokens if t.upper() not in INVALID_TOKENS]

    # Fallback to DOGE for empty tokens (Musk's vague crypto tweets)
    if not tokens:
        tokens = [DEFAULT_TOKEN]

    # Priority order based on Musk's influence
    priority_order = ['DOGE', 'BTC', 'ETH', 'SHIB', 'FLOKI', 'XRP', 'ADA', 'LINK', 'MATIC']
    primary_token = None
    for t in priority_order:
        if t in tokens:
            primary_token = t
            break
    if not primary_token and tokens:
        primary_token = tokens[0]

    if not primary_token or primary_token not in token_prices:
        return features

    features['price_token'] = primary_token
    price_df = token_prices[primary_token]

    # Extract prices at each timepoint
    timepoints = {
        'price_before_24h': tweet_time - timedelta(hours=24),
        'price_before_1h': tweet_time - timedelta(hours=1),
        'price_at_tweet': tweet_time,
        'price_after_1h': tweet_time + timedelta(hours=1),
        'price_after_24h': tweet_time + timedelta(hours=24),
        'price_after_7d': tweet_time + timedelta(days=7),
    }

    for feature_name, target_time in timepoints.items():
        features[feature_name] = get_price_at_time(price_df, target_time)

    # Get volume at tweet time
    if not price_df.empty:
        tweet_time_aware = tweet_time if tweet_time.tzinfo else tweet_time.tz_localize('UTC')
        time_diffs = abs(price_df['timestamp'] - tweet_time_aware)
        min_idx = time_diffs.idxmin()
        if time_diffs[min_idx] < timedelta(hours=24):
            features['volume_at_tweet'] = price_df.loc[min_idx, 'volume']

    # Calculate percentage changes
    if features['price_at_tweet'] and features['price_at_tweet'] > 0:
        p_at = features['price_at_tweet']

        if features['price_after_1h']:
            features['pct_change_1h'] = ((features['price_after_1h'] - p_at) / p_at) * 100

        if features['price_after_24h']:
            features['pct_change_24h'] = ((features['price_after_24h'] - p_at) / p_at) * 100

        if features['price_after_7d']:
            features['pct_change_7d'] = ((features['price_after_7d'] - p_at) / p_at) * 100

    return features


def process_tweets_with_prices(input_path: str, output_path: str):
    """
    Main function to process tweets and add price features.
    """
    if not test_api_connection():
        print("\n⚠️  API connection failed. Check your .env file:")
        print("   CRYPTOCOMPARE_API_KEY=your_key")
        return None

    print("\nLoading tweet data...")
    df = pd.read_csv(input_path)

    df_crypto = df[df['is_crypto_related'] == True].copy()
    print(f"Processing {len(df_crypto):,} crypto-related tweets")

    df_crypto['createdAt'] = pd.to_datetime(df_crypto['createdAt'], utc=True)

    # Get unique tokens mentioned
    all_tokens = set()
    for tokens in df_crypto['crypto_tokens'].dropna():
        if isinstance(tokens, str):
            try:
                tokens = eval(tokens)
            except:
                continue
        if isinstance(tokens, list):
            all_tokens.update([t.upper() for t in tokens])

    # Filter out invalid tokens (categories)
    all_tokens = {t for t in all_tokens if t not in INVALID_TOKENS}

    # Always include DOGE (default fallback)
    all_tokens.add(DEFAULT_TOKEN)

    # Only keep supported tokens
    all_tokens = {t for t in all_tokens if t in SUPPORTED_TOKENS}

    print(f"\nTokens to fetch: {all_tokens}")

    min_date = df_crypto['createdAt'].min()
    max_date = df_crypto['createdAt'].max()
    print(f"Tweet date range: {min_date.date()} to {max_date.date()}")

    # Fetch all token prices
    print("\n" + "="*50)
    print("FETCHING PRICE DATA FROM CRYPTOCOMPARE")
    print("="*50)
    token_prices = fetch_all_token_prices(list(all_tokens), min_date, max_date)

    # Extract price features
    print("\n" + "="*50)
    print("EXTRACTING PRICE FEATURES")
    print("="*50)

    price_features = []
    for idx, row in tqdm(df_crypto.iterrows(), total=len(df_crypto), desc="Processing tweets"):
        features = extract_price_features(row, token_prices)
        price_features.append(features)

    price_df = pd.DataFrame(price_features)
    for col in price_df.columns:
        df_crypto[col] = price_df[col].values

    # Save
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df_crypto.to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")

    # Summary
    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total tweets: {len(df_crypto):,}")
    print(f"Tweets with price data: {df_crypto['price_at_tweet'].notna().sum():,}")

    print(f"\nPrice features coverage:")
    price_cols = ['price_before_24h', 'price_before_1h', 'price_at_tweet',
                  'price_after_1h', 'price_after_24h', 'price_after_7d']
    for col in price_cols:
        coverage = df_crypto[col].notna().sum()
        pct = 100 * coverage / len(df_crypto)
        print(f"  {col}: {coverage:,} ({pct:.1f}%)")

    print(f"\nTokens analyzed:")
    print(df_crypto['price_token'].value_counts().to_string())

    # Count fallback usage
    empty_token_mask = df_crypto['crypto_tokens'].apply(
        lambda x: x == '[]' or x == [] or pd.isna(x)
    )
    fallback_count = (empty_token_mask & (df_crypto['price_token'] == DEFAULT_TOKEN)).sum()
    print(f"\nDOGE fallback used for empty crypto_tokens: {fallback_count:,} tweets")

    print(f"\nAverage price changes after Musk tweets:")
    for col in ['pct_change_1h', 'pct_change_24h', 'pct_change_7d']:
        mean_change = df_crypto[col].mean()
        if pd.notna(mean_change):
            print(f"  {col}: {mean_change:+.2f}%")

    return df_crypto

In [16]:
input_file = "../data/processed/musk_tweets_llm_features.csv"
output_file = "../data/curated/musk_crypto_tweets_final.csv"

df = process_tweets_with_prices(input_file, output_file)
print("\nDone!")


Testing CryptoCompare API connection...
  API Key set: Yes
  ✓ API connection OK

Loading tweet data...
Processing 1,003 crypto-related tweets

Tokens to fetch: {'SHIB', 'DOGE', 'FTT', 'FLOKI', 'BTC', 'MATIC', 'ADA', 'LINK', 'ETH', 'XRP', 'BABYDOGE'}
Tweet date range: 2012-02-06 to 2025-04-13

FETCHING PRICE DATA FROM CRYPTOCOMPARE


Fetching token prices:   0%|          | 0/11 [00:00<?, ?it/s]


Fetching SHIB...
  Got 1,400 data points from 2021-06-22 to 2025-04-21


Fetching token prices:   9%|▉         | 1/11 [00:07<01:19,  7.95s/it]


Fetching DOGE...
  Got 4,098 data points from 2014-02-01 to 2025-04-21


...


Fetching BABYDOGE...
  Got 412 data points from 2024-03-06 to 2025-04-21


Fetching token prices: 100%|██████████| 11/11 [01:49<00:00,  9.98s/it]



EXTRACTING PRICE FEATURES


Processing tweets: 100%|██████████| 1003/1003 [00:04<00:00, 237.25it/s]


Saved to ../data/curated/musk_crypto_tweets_final.csv

SUMMARY
Total tweets: 1,003
Tweets with price data: 998

Price features coverage:
  price_before_24h: 998 (99.5%)
  price_before_1h: 998 (99.5%)
  price_at_tweet: 998 (99.5%)
  price_after_1h: 998 (99.5%)
  price_after_24h: 998 (99.5%)
  price_after_7d: 998 (99.5%)

Tokens analyzed:
price_token
DOGE        946
BTC          45
ETH           6
SHIB          3
XRP           1
FTT           1
BABYDOGE      1

DOGE fallback used for empty crypto_tokens: 229 tweets

Average price changes after Musk tweets:
  pct_change_1h: -0.02%
  pct_change_24h: -0.36%
  pct_change_7d: -2.02%

Done!
